# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading and exploring the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema, accessible at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset package using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Metadata is exposed as an object; display main properties
md = dataset.metadata
print(f"Dataset Title: {md.name}")
print(f"Identifier: {md.identifier}")
print("\nDescription:")
print(md.description)
print("\nKeywords:", getattr(md, 'keywords', 'N/A'))

## 2. Data Overview
Review available record sets (tables), their `@id` identifiers, and the field/column structure for each record set.

Below, we list all available record sets, their IDs, and the fields within each set. All references are made using `@id`.

Note: You can always cross-reference with the Croissant schema JSON-LD for the authoritative structure. For this dataset, we let `mlcroissant` enumerate the available record sets and fields.

In [ ]:
def print_record_sets(dataset):
    print("Found the following record sets:\n")
    for rs in dataset.record_sets:
        print(f"@id: {rs['@id']}")
        print(f"  Name: {rs.get('name')}")
        print(f"  Description: {rs.get('description','')}")
        fields = rs.get('field', [])
        if not isinstance(fields, list):
            fields = [fields]
        if fields:
            print(f"  Fields (by @id):")
            for field in fields:
                if isinstance(field, dict):
                    print(f"    - {field.get('@id')}")
                else:
                    print(f"    - {field}")
        print("")
print_record_sets(dataset)

Let's print a few records from each record set for inspection. Note that in `mlcroissant`, you reference the record set using its `@id`.

In [ ]:
# Print first 2 records for each record set, using @id
for rs in dataset.record_sets:
    rs_id = rs['@id']
    print(f"--- Record set @id: {rs_id} ---")
    try:
        for i, record in enumerate(dataset.records(record_set=rs_id)):
            print(record)
            if i >= 1:
                break
    except Exception as e:
        print(f"  (No records or error: {e})")
    print()

## 3. Data Extraction
Load data from selected record sets into pandas DataFrames for analysis. Here, each record set is referenced by its `@id`. We show the fields/columns for the main data record set.

In [ ]:
# List all record set @ids
record_sets = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

# Load each table into a DataFrame
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for '{rs_id}'. Fields: {df.columns.tolist()}")
    else:
        print(f"No records found for '{rs_id}'.")

# For demonstration, preview the first record set found
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in primary record set({main_rs_id}):\n", dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()
else:
    print("No dataframes loaded!")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering and normalizing numeric fields. All references are made using field/column `@id` names, directly as they appear in the DataFrame columns.

In [ ]:
import numpy as np

# EDIT THIS: Choose the main record set for analysis
record_set_id = main_rs_id  # From prior code cell
df = dataframes[record_set_id]

# Inspect available columns and pick a numeric column by @id (e.g. 'coverage', 'molecular_weight', etc.)
print("Available columns:", df.columns.tolist())
# Try to auto-select one numeric field, else fallback
numeric_fields = df.select_dtypes(include=np.number).columns.tolist()
if numeric_fields:
    numeric_field = numeric_fields[0]
else:
    # If all fields are object type, forcibly convert a likely numeric field
    guess_candidates = [col for col in df.columns if 'coverage' in col.lower() or 'mw' in col.lower() or 'weight' in col.lower()]
    numeric_field = guess_candidates[0] if guess_candidates else df.columns[0]
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

print(f"Using numeric field: {numeric_field}")

# Remove possible NaNs
filtered_df = df[df[numeric_field].notnull()].copy()

# Filter: keep only records with that field > threshold
threshold = filtered_df[numeric_field].mean() if len(filtered_df) else 0
filtered_df = filtered_df[filtered_df[numeric_field] > threshold]

print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {filtered_df.shape[0]} left")
print(filtered_df.head())

# Normalize
col_norm = f"{numeric_field}_normalized"
filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, col_norm]].head())

# Group by a likely annotation field if available (e.g. protein or description)
group_fields = [col for col in df.columns if any(g in col.lower() for g in ['protein', 'description', 'type'])]
if group_fields:
    group_field = group_fields[0]
    print(f"\nGrouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(grouped_df.head())

## 5. Visualization
Visualize the distributions and trends within the numeric field. The plots use the field `@id` as axis/label for clarity and reproducibility.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 4))
sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If grouped
if 'grouped_df' in locals():
    grouped_df = grouped_df.sort_values(numeric_field, ascending=False)
    plt.figure(figsize=(12, 5))
    grouped_df[numeric_field].head(10).plot(kind='bar')
    plt.title(f"Top 10 {group_field} by mean {numeric_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load, explore, and analyze the FAIR^2 dataset describing mass spectrometry analysis of extracellular vesicles from stimulated human mast cells. We inspected the dataset structure via record set and field `@id`s, loaded tables into DataFrames, filtered and normalized numeric data, and produced insight through quick visualizations.

To adapt this workflow to future Croissant datasets, always reference tables and fields by their `@id`, and begin with loading the metadata to guide further exploration. For in-depth analysis, consult the schema and documentation for precise field semantics.

Further work can involve deeper validation, combining record sets, or domain-specific analyses based on the dataset's biomedical context.